In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
from scipy.io import loadmat
from PIL import Image
from transformers import ViTModel, ViTConfig
from tqdm import tqdm

In [3]:
# 设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 图像数据集路径
image_root = "L:/常惠林/萎凋/自然萎凋/原始"

# 图像预处理
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=30),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.ToTensor(),
    transforms.GaussianBlur(5),
    transforms.RandomHorizontalFlip(),
])

In [4]:
class MultimodalDataset(Dataset):
    def __init__(self, image_root, nir_path, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform

        for i, cls in enumerate(["class_1", "class_2", "class_3"]):
            folder = os.path.join(image_root, cls)
            for fname in sorted(os.listdir(folder)):
                self.image_paths.append(os.path.join(folder, fname))
                self.labels.append(i)

        nir_data = loadmat(nir_path)['nir']  # 假设键为'nir'
        self.nir_data = torch.tensor(nir_data, dtype=torch.float32)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        nir = self.nir_data[idx]
        label = self.labels[idx]
        return image, nir, label

# 简单的1D CNN用于NIR编码器
class NIREncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, L)
        x = self.net(x)
        return x.squeeze(-1)

In [ ]:
# 多模态分类器
class MultimodalClassifier(nn.Module):
    def __init__(self, img_encoder, nir_encoder, hidden_dim=768):
        super().__init__()
        self.img_encoder = img_encoder
        self.nir_encoder = nir_encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 32, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, img, nir):
        img_feat = self.img_encoder(pixel_values=img).last_hidden_state[:, 0, :]  # CLS token
        nir_feat = self.nir_encoder(nir)
        fused = torch.cat([img_feat, nir_feat], dim=1)
        return self.classifier(fused), img_feat


In [ ]:
# 训练函数
def train_epoch(model, teacher, dataloader, criterion_cls, criterion_kd, optimizer):
    model.train()
    total_loss, total_correct = 0, 0
    for img, nir, label in tqdm(dataloader):
        img, nir, label = img.to(device), nir.to(device), label.to(device)

        with torch.no_grad():
            teacher_feat = teacher(img).last_hidden_state[:, 0, :]

        out, student_feat = model(img, nir)
        loss_cls = criterion_cls(out, label)
        loss_kd = criterion_kd(student_feat, teacher_feat)
        loss = loss_cls + 0.5 * loss_kd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * label.size(0)
        total_correct += (out.argmax(1) == label).sum().item()

    return total_loss / len(dataloader.dataset), total_correct / len(dataloader.dataset)


In [ ]:
# 评估 + 可视化
def evaluate(model, dataloader):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    with torch.no_grad():
        for img, nir, label in dataloader:
            img, nir = img.to(device), nir.to(device)
            out, _ = model(img, nir)
            probs = F.softmax(out, dim=1)
            y_scores.append(probs.cpu().numpy())
            y_pred.append(probs.argmax(1).cpu().numpy())
            y_true.append(label.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_scores = np.concatenate(y_scores)

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title("Confusion Matrix")
    plt.show()

    # ROC曲线
    plt.figure()
    for i in range(3):
        fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.show()

In [ ]:
# 主流程
if __name__ == '__main__':
    dataset = MultimodalDataset(image_root, 'L:/常惠林/萎凋/NIR.mat', transform=transform)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

    teacher_vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
    student_vit_config = ViTConfig(
        hidden_size=384,
        num_hidden_layers=6,
        num_attention_heads=6,
        intermediate_size=384*4,
        image_size=224,
        patch_size=16
    )
    student_vit = ViTModel(student_vit_config).to(device)

    nir_encoder = NIREncoder().to(device)
    model = MultimodalClassifier(student_vit, nir_encoder, hidden_dim=384).to(device)

    criterion_cls = nn.CrossEntropyLoss()
    criterion_kd = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(10):
        loss, acc = train_epoch(model, teacher_vit, dataloader, criterion_cls, criterion_kd, optimizer)
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Acc: {acc:.4f}")

    evaluate(model, dataloader)